# 🚀 YOLOv5 CPU vs GPU Benchmark
### Real-Time Object Detection System — Performance Analysis

**Before running:** Make sure you have enabled the GPU runtime!
- Click **Runtime** in the top menu
- Click **Change runtime type**
- Set Hardware accelerator to **T4 GPU**
- Click **Save**
- Then run all cells from top to bottom


## Step 1: Check GPU is available

In [ ]:
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('ERROR: GPU not detected! Go to Runtime > Change runtime type > T4 GPU')

## Step 2: Install dependencies

In [ ]:
!pip install -q ultralytics opencv-python-headless numpy

## Step 3: Download a sample test image

In [ ]:
import urllib.request
import os

# Download a sample image from COCO dataset
os.makedirs('data/sample_images', exist_ok=True)
url = 'https://ultralytics.com/images/zidane.jpg'
urllib.request.urlretrieve(url, 'data/sample_images/test.jpg')
print('Test image downloaded successfully!')

# Display the image
from IPython.display import Image as IPImage
IPImage('data/sample_images/test.jpg', width=400)

## Step 4: Load YOLOv5 model

In [ ]:
import torch
import cv2
import time
import numpy as np

print('Loading YOLOv5s model...')
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)
print('Model loaded successfully!')

## Step 5: Define benchmark function

In [ ]:
def run_benchmark(model, image_path, device, num_iterations=100):
    """
    Benchmark YOLOv5 inference on a given device.
    Returns dictionary of performance statistics.
    """
    model.to(device)
    model.eval()

    # Read image
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    inference_times = []

    # Warm-up runs (first few runs are always slower)
    print(f'  Running 5 warm-up iterations on {device.upper()}...')
    for _ in range(5):
        with torch.no_grad():
            _ = model(img_rgb)

    # Actual benchmark
    print(f'  Running {num_iterations} benchmark iterations on {device.upper()}...')
    for i in range(num_iterations):
        start = time.time()
        with torch.no_grad():
            results = model(img_rgb)
        end = time.time()
        inference_times.append((end - start) * 1000)  # convert to ms

        if (i + 1) % 25 == 0:
            print(f'    Completed {i + 1}/{num_iterations} iterations')

    return {
        'device': device.upper(),
        'mean':   round(float(np.mean(inference_times)), 2),
        'median': round(float(np.median(inference_times)), 2),
        'std':    round(float(np.std(inference_times)), 2),
        'min':    round(float(np.min(inference_times)), 2),
        'max':    round(float(np.max(inference_times)), 2),
        'p95':    round(float(np.percentile(inference_times, 95)), 2),
        'p99':    round(float(np.percentile(inference_times, 99)), 2),
    }

print('Benchmark function defined!')

## Step 6: Run CPU benchmark

In [ ]:
print('=' * 50)
print('BENCHMARKING ON CPU')
print('=' * 50)

cpu_results = run_benchmark(model, 'data/sample_images/test.jpg', 'cpu', num_iterations=100)

print(f"\nCPU Results:")
print(f"  Mean inference time:   {cpu_results['mean']} ms")
print(f"  Median inference time: {cpu_results['median']} ms")
print(f"  Std deviation:         {cpu_results['std']} ms")
print(f"  Min inference time:    {cpu_results['min']} ms")
print(f"  Max inference time:    {cpu_results['max']} ms")
print(f"  95th percentile:       {cpu_results['p95']} ms")
print(f"  99th percentile:       {cpu_results['p99']} ms")

## Step 7: Run GPU benchmark

In [ ]:
if torch.cuda.is_available():
    print('=' * 50)
    print('BENCHMARKING ON GPU (CUDA)')
    print('=' * 50)

    gpu_results = run_benchmark(model, 'data/sample_images/test.jpg', 'cuda', num_iterations=100)

    print(f"\nGPU Results:")
    print(f"  Mean inference time:   {gpu_results['mean']} ms")
    print(f"  Median inference time: {gpu_results['median']} ms")
    print(f"  Std deviation:         {gpu_results['std']} ms")
    print(f"  Min inference time:    {gpu_results['min']} ms")
    print(f"  Max inference time:    {gpu_results['max']} ms")
    print(f"  95th percentile:       {gpu_results['p95']} ms")
    print(f"  99th percentile:       {gpu_results['p99']} ms")
else:
    print('ERROR: GPU not available! Make sure you enabled T4 GPU in runtime settings.')

## Step 8: Compare results and calculate speedup

In [ ]:
if torch.cuda.is_available():
    speedup_mean   = round(cpu_results['mean']   / gpu_results['mean'], 2)
    speedup_median = round(cpu_results['median'] / gpu_results['median'], 2)
    speedup_p95    = round(cpu_results['p95']    / gpu_results['p95'], 2)

    print('=' * 60)
    print('FINAL BENCHMARK RESULTS — SAVE THESE NUMBERS!')
    print('=' * 60)
    print(f'\n  CPU mean inference:    {cpu_results["mean"]} ms')
    print(f'  GPU mean inference:    {gpu_results["mean"]} ms')
    print(f'  CPU p95 latency:       {cpu_results["p95"]} ms')
    print(f'  GPU p95 latency:       {gpu_results["p95"]} ms')
    print(f'\n  Mean speedup:          {speedup_mean}x')
    print(f'  Median speedup:        {speedup_median}x')
    print(f'  p95 speedup:           {speedup_p95}x')
    print('\n' + '=' * 60)
    print(f'  GPU IS {speedup_mean}x FASTER THAN CPU (mean inference)')
    print('=' * 60)
    print('\n✅ Copy these numbers into your README and resume!')
else:
    print('GPU not available — only CPU results recorded.')

## Step 9: Save results to JSON

In [ ]:
import json

all_results = {'cpu': cpu_results}
if torch.cuda.is_available():
    all_results['cuda'] = gpu_results
    all_results['speedup'] = {
        'mean':   speedup_mean,
        'median': speedup_median,
        'p95':    speedup_p95
    }

with open('benchmark_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print('Results saved to benchmark_results.json')
print('Download this file from the Files panel on the left sidebar!')
print(json.dumps(all_results, indent=2))